In [ ]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber

In [ ]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [ ]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/temas_ejercicio.json", "r") as file:
    temas_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [ ]:
# file = Path("enlace_químico/2022 - Enlace Químico.pdf")

In [ ]:
def extract_statement(exercise):
    m = re.search("R E S O L U C I Ó N", exercise)
    if m:
        end_pos = m.start()
        exercise = exercise[:end_pos].strip()
        lines = exercise.split("\n")
        exam = lines[-1]
        statement = "\n".join(lines[:-1])
        return exam, statement
    return None

In [ ]:
def extract_exercises(file):
    exercises = []
    with pdfplumber.open(file) as pdf:
        for idx, page in enumerate(pdf.pages):
            texto = ext.extract_text_from_page(page)
            if idx > 0:  # ejercicios comienzan en 2da página
                st = extract_statement(texto)
                if st:
                    exercises.append(extract_statement(texto))
    return exercises

In [ ]:
exercises_list = []
for file in Path("enlace_químico").iterdir():   
    exams, statements = [], []
    #print("Processing file: ", file.name)
    exercises = extract_exercises(file)
    for ex in exercises:
        if ex:
            exercises_list.append({"exam" : ex[0], "statement": ex[1]})

In [ ]:
df = pd.DataFrame(exercises_list)

In [ ]:
def classify_exercise(statement: str, topic: str, keywords = quimica_keywords) -> tuple:
    topic_keywords = keywords[topic]["keywords_tema"]

    exercise_topic = None
    for topic_kw in topic_keywords:
        if topic_kw in statement.lower():
            exercise_topic = topic    
            break

    if exercise_topic:
        exercise_types = []
        exercise_type_keywords = keywords[topic]["tipos"]
        
        for tipo, tipo_keywords in exercise_type_keywords.items():
            if any(kw in statement.lower() for kw in tipo_keywords):
                exercise_types.append(tipo)
    else:
        exercise_topic, exercise_types = None, [] 

    return exercise_topic, exercise_types

In [ ]:
df.head()

In [ ]:
df["topic"], df["exercise_type"] = zip(*df.statement.apply(classify_exercise, topic="enlace_químico"))

In [ ]:
df.isna().sum()

In [ ]:
df = df.explode("exercise_type")
df = df.sort_values("exam").reset_index(drop=True)
df = df.drop_duplicates()

In [ ]:
# df

In [ ]:
df.exercise_type.unique()